In [1]:
import torch
import pandas as pd
import numpy as np

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import classification_report, confusion_matrix
import torch.nn.functional as F

In [2]:
# Load Model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSequenceClassification.from_pretrained(
    "../models/distilbert_v2"
)

model.to(device)
model.eval()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [3]:
# Load Dataset

df = pd.read_csv("../data/processed/reviews_cleaned.csv")

df.head()

,review_text,label,rating,source,combined_text
0,earn money fast and win exciting prizes!!!!!!,1,3,clean,earn money fast and win exciting prizes!!!!!!....
1,visit this link to get this product now!!!,1,3,clean,visit this link to get this product now!!!
2,"This laptop is okay, it works well.",0,4,clean,"This laptop is okay, it works well."
3,"THIS LAPTOP IS OKAY, IT STOPPED WORKING. THIS ...",0,4,clean,"THIS LAPTOP IS OKAY, IT STOPPED WORKING. THIS ..."
4,I bought this camera recently and it stopped w...,0,2,clean,I bought this camera recently and it stopped w...


In [4]:
# Fix label mapping

def map_label(x):
    x = str(x).lower()
    if x == "fake":
        return 1
    elif x == "genuine":
        return 0
    else:
        return int(x)

df["label"] = df["label"].apply(map_label)

print(df["label"].unique())

[1 0]


In [5]:
# Tokenization 
df = df.dropna(subset=["combined_text"])

df["combined_text"] = df["combined_text"].astype(str)

df = df[df["combined_text"].str.strip() != ""]

print("Clean dataset size:", len(df))
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

encodings = tokenizer(
    df["combined_text"].tolist(),
    truncation=True,
    padding=True,
    max_length=64
)

Clean dataset size: 10000


In [6]:
# Dataset 

class ReviewDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [7]:
# DataLoader

dataset = ReviewDataset(encodings, df["label"])

loader = DataLoader(dataset, batch_size=8, num_workers=0)

In [8]:
# Evaluation 

all_preds = []
all_labels = []
confidences = []

correct = 0
total = 0

with torch.no_grad():
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)

        preds = torch.argmax(outputs.logits, dim=1)

        correct += (preds == batch["labels"]).sum().item()
        total += len(preds)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch["labels"].cpu().numpy())

        probs = F.softmax(outputs.logits, dim=1)
        max_conf = torch.max(probs, dim=1).values
        confidences.extend(max_conf.cpu().numpy())

accuracy = correct / total

print(f"\nAccuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=["Genuine", "Fake"]))

print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

print("\nConfidence Stats:")
print(f"Average Confidence: {np.mean(confidences):.4f}")


Accuracy: 0.9860

Classification Report:
              precision    recall  f1-score   support

     Genuine       0.97      1.00      0.98      4309
        Fake       1.00      0.98      0.99      5691

    accuracy                           0.99     10000
   macro avg       0.98      0.99      0.99     10000
weighted avg       0.99      0.99      0.99     10000


Confusion Matrix:
[[4302    7]
 [ 133 5558]]

Confidence Stats:
Average Confidence: 0.9624


In [9]:
# Demo Predictions

samples = df.sample(5)

for i, row in samples.iterrows():
    text = f"Rating: {row['rating']} | Review: {row['review_text']}"

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=64)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()
        confidence = torch.max(probs).item()

    print("\n-----------------------------")
    print("Review:", row["review_text"])
    print("Actual:", "Fake" if row["label"] == 1 else "Genuine")
    print("Predicted:", "Fake" if pred == 1 else "Genuine")
    print("Confidence:", f"{confidence:.2f}")


-----------------------------
Review: PHONE IS UNBELIEVABLE!!! DON'T MISS OUT!!!
Actual: Fake
Predicted: Fake
Confidence: 1.00

-----------------------------
Review: THIS WATCH IS OKAY, IT POOR QUALITY.
Actual: Genuine
Predicted: Genuine
Confidence: 1.00

-----------------------------
Review: THE CAMERA POOR QUALITY.
Actual: Genuine
Predicted: Genuine
Confidence: 1.00

-----------------------------
Review: LIMITED OFFER!!! LIMITED TIME OFFER!!! LIMITED OFFER!!! LIMITED TIME OFFER!!!
Actual: Fake
Predicted: Fake
Confidence: 1.00

-----------------------------
Review: Just to start, I am not an expert on swimming. I did not know that the person who gave me this swimsuit said it was designed to be worn with a cap. I was amazed at the swimsuit's swim quality. I was also amazed at how well it held up. I had a 2 day swim at the pool and I had to wear it with a cap. I will say I am very pleased with it. I am very pleased with it.This is a great tool for camping or backpacking. The mount is v

In [12]:
# DEMO

rating = input("Enter rating (1-5): ")
review = input("Enter review text: ")

text = f"Rating: {rating} | Review: {review}"

# Tokenize
inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=64
)

inputs = {k: v.to(device) for k, v in inputs.items()}

# Predict
with torch.no_grad():
    outputs = model(**inputs)
    probs = F.softmax(outputs.logits, dim=1)

pred = torch.argmax(probs, dim=1).item()
confidence = torch.max(probs).item()

# Output
print("\nCustom Review Analysis")
print("Input:", text)
print("Prediction:", "Fake" if pred == 1 else "Genuine")
print("Confidence:", f"{confidence:.2f}")


Custom Review Analysis
Input: Rating: 5 | Review: Great product! If you want to make $5000 a week working from home, click here:
Prediction: Fake
Confidence: 0.68


### Model Summary
* **Model:** DistilBERT (pretrained transformer)
* **Task:** Fake Review Detection
* **Input:** Combined Rating + Review Text
* **Output:**
  * `0` → Genuine
  * `1` → Fake

### Training
* **Epochs:** 1
* **Batch size:** 16
* **Optimizer:** AdamW

### Performance
* **Validation Accuracy:** 98%

### Observations
* Model captures exaggerated language patterns.
* Rating + review combination improves detection
* Model is suitable for moderation systems like TrustNet